In [ ]:
!pip uninstall -y mediapipe gradio
!pip install mediapipe gradio scikit-learn matplotlib opencv-python-headless

Found existing installation: gradio 6.19.0
Uninstalling gradio-6.19.0:
  Successfully uninstalled gradio-6.19.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 77.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.0/31.0 MB 59.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.4/137.4 kB 10.1 MB/s eta 0:00:00
  Attempting uninstall: absl-py
    Found existing installation: absl-py 1.4.0
    Uninstalling absl-py-1.4.0:
      Successfully uninstalled absl-py-1.4.0


In [ ]:
!wget -q -O face_landmarker.task https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task

In [ ]:
import cv2
import gradio as gr
import matplotlib
import matplotlib.pyplot as plt
import mediapipe as mp
import numpy as np
from PIL import Image
from sklearn.cluster import KMeans

from mediapipe.tasks import python
from mediapipe.tasks.python import vision

# Force Matplotlib to use a non-interactive backend to avoid threading memory leaks in web servers
matplotlib.use("Agg")

# ==========================================
# 1. INITIALIZE MEDIAPIPE FACE LANDMARKER (updated for new tasks API)
#    NOTE: You need to download the 'face_landmarker.task' model file.
#    Run `!wget -q -O face_landmarker.task https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task`
#    in a separate cell before running this code.
# ==========================================
base_options = python.BaseOptions(model_asset_path='face_landmarker.task')
options = vision.FaceLandmarkerOptions(base_options=base_options,
                                       output_face_blendshapes=False,
                                       output_facial_transformation_matrixes=False,
                                       running_mode=vision.RunningMode.IMAGE,
                                       num_faces=1)
face_landmarker = vision.FaceLandmarker.create_from_options(options)

# ==========================================
# 2. DEFINE THE COLOR RECOMMENDATION DATA
# ==========================================
SEASON_PALETTES = {
    "Spring": {
        "colors": [
            ("#F5A623", "Warm Coral"),
            ("#FFF3A7", "Butter Yellow"),
            ("#A3E4D7", "Mint Green"),
            ("#F1948A", "Peach"),
            ("#5DADE2", "Sky Blue"),
        ],
        "avoid": ["Pure Black", "Icy Grey", "Deep Burgundy"],
        "jewelry": "Bright Gold / Rose Gold",
        "hair": "Honey Blonde, Golden Brown, Copper",
        "makeup": "Peach blush, Warm nude lips, Champagne highlighter",
    },
    "Autumn": {
        "colors": [
            ("#808000", "Olive Green"),
            ("#B87333", "Copper/Rust"),
            ("#C19A6B", "Camel"),
            ("#E65C00", "Burnt Orange"),
            ("#4A5D4E", "Forest Green"),
        ],
        "avoid": ["Cool Grey", "Pure White", "Icy Blue"],
        "jewelry": "Antique Gold / Brass",
        "hair": "Rich Auburn, Chocolate Brown, Golden Shimmer",
        "makeup": "Terracotta blush, Brick red/Warm brown lips",
    },
    "Summer": {
        "colors": [
            ("#D98880", "Dusty Pink"),
            ("#7FB3D5", "Slate Blue"),
            ("#BB8FCE", "Lavender"),
            ("#A9DFBF", "Soft Sage"),
            ("#E5E7E9", "Cool Grey"),
        ],
        "avoid": ["Neon Orange", "Mustard Yellow", "Deep Earth Browns"],
        "jewelry": "Silver / White Gold",
        "hair": "Ash Blonde, Ash Brown, Platinum accents",
        "makeup": "Cool pink blush, Berry/Mauve lip tints",
    },
    "Winter": {
        "colors": [
            ("#000080", "Navy Blue"),
            ("#50C878", "Emerald Green"),
            ("#000000", "Pure Black"),
            ("#FFFFFF", "Pure White"),
            ("#C70039", "Ruby Red"),
        ],
        "avoid": ["Golden Brown", "Orange-Red", "Pastel Beige"],
        "jewelry": "Bright Silver / Platinum",
        "hair": "Jet Black, Dark Ash Brown, Silver/White hair",
        "makeup": "Plum or True Red lipstick, Crisp highlighter",
    },
}


# ==========================================
# 3. CORE EXTRACTION AND PROCESSING PIPELINE
# ==========================================
def analyze_personal_color(image):
    if image is None:
        return "Please upload an image.", None

    # Convert PIL Image or numpy array to RGB safely
    img = np.array(image)
    if img.ndim != 3 or img.shape[2] < 3:
        return "Invalid image format. Please upload a clear color photo.", None

    img = img[:, :, :3]  # Strip Alpha channel if it exists
    h, w, c = img.shape

    # Convert numpy array to MediaPipe Image
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=img)

    # Process image with MediaPipe Face Landmarker
    face_landmarker_result = face_landmarker.detect(mp_image)

    if not face_landmarker_result.face_landmarks:
        return (
            "No face detected. Please ensure your face is clear, forward-facing, and well-lit.",
            None,
        )

    landmarks = face_landmarker_result.face_landmarks[0]

    # MediaPipe Face Mesh Indices for safe skin sampling (Cheeks and Forehead)
    safe_zone_indices = [
        70,
        71,
        139,
        368,
        399,
        412,  # Forehead regions
        117,
        118,
        119,
        123,
        142,  # Left Cheek
        346,
        347,
        348,
        352,
        371,  # Right Cheek
    ]

    # Extract pixel coordinates of our safe zones
    skin_pixels = []
    for idx in safe_zone_indices:
        lm = landmarks[idx]
        cx, cy = int(lm.x * w), int(lm.y * h)

        # Add a 5x5 bounding box around each landmark to grab a cluster of pixels
        for dx in range(-2, 3):
            for dy in range(-2, 3):
                nx, ny = cx + dx, cy + dy
                if 0 <= nx < w and 0 <= ny < h:
                    skin_pixels.append(img[ny, nx])

    skin_pixels = np.array(skin_pixels)

    if len(skin_pixels) == 0:
        return "Failed to sample skin pixels accurately.", None

    # Use KMeans to find the dominant skin color (ignoring minor shadow variations)
    kmeans = KMeans(n_clusters=3, n_init=10, random_state=42)
    kmeans.fit(skin_pixels)

    # Find the largest cluster (dominant skin tone)
    labels, counts = np.unique(kmeans.labels_, return_counts=True)
    dominant_rgb = kmeans.cluster_centers_[labels[np.argmax(counts)]].astype(
        np.uint8
    )

    # Convert dominant RGB to LAB for precise structural analysis
    rgb_pixel = np.uint8([[dominant_rgb]])
    lab_pixel = cv2.cvtColor(rgb_pixel, cv2.COLOR_RGB2LAB)[0][0]

    L, A, B = int(lab_pixel[0]), int(lab_pixel[1]), int(lab_pixel[2])

    # ==========================================
    # 4. UNDERTONE & SEASON HEURISTIC ENGINE
    # ==========================================
    # Adjusted L thresholds to account for natural exposure variances in standard photos
    if L > 165:
        skin_tone = "Fair / Light"
    elif L > 115:
        skin_tone = "Medium"
    else:
        skin_tone = "Deep / Dark"

    # Delta offset calculation (B channel natively shifted by 128 in OpenCV uint8 conversion)
    warmth_score = B - 128

    # Calculate system confidence metric
    confidence = min(98, int(70 + (abs(warmth_score) * 2.5)))

    if warmth_score > 3:
        undertone = "Warm"
        season = "Spring" if L > 140 else "Autumn"
    elif warmth_score < -3:
        undertone = "Cool"
        season = "Summer" if L > 140 else "Winter"
    else:
        undertone = "Neutral"
        # Balance out boundary states
        season = "Autumn" if L < 130 else "Summer"
        confidence = 85

    palette_data = SEASON_PALETTES[season]

    # ==========================================
    # 5. GENERATE VISUAL PALETTE DIAGRAM
    # ==========================================
    fig, ax = plt.subplots(figsize=(7, 2.2))
    ax.set_xlim(0, 5)
    ax.set_ylim(0, 1)
    ax.axis("off")

    for i, (hex_code, name) in enumerate(palette_data["colors"]):
        ax.add_patch(plt.Rectangle((i, 0), 1, 1, color=hex_code))

        # Dynamically evaluate text color contrast safely inside OpenCV
        rgb_color = (np.array(matplotlib.colors.to_rgb(hex_code)) * 255).astype(
            np.uint8
        )
        test_pixel = np.uint8([[rgb_color]])
        text_L = cv2.cvtColor(test_pixel, cv2.COLOR_RGB2LAB)[0][0][0]
        text_color = "white" if text_L < 125 else "black"

        ax.text(
            i + 0.5,
            0.5,
            name,
            color=text_color,
            fontsize=10,
            weight="bold",
            ha="center",
            va="center",
        )

    plt.tight_layout()

    # Thread-safe conversion of Matplotlib canvas back into PIL
    fig.canvas.draw()
    rgba_buffer = fig.canvas.buffer_rgba()
    palette_img = Image.frombuffer(
        "RGBA", fig.canvas.get_width_height(), rgba_buffer, "raw", "RGBA", 0, 1
    ).convert("RGB")
    plt.close(fig)

    # Format Markdown Output
    output_text = f"""
### 📊 Analysis Results
* **Skin Tone Depth:** {skin_tone}
* **Undertone:** {undertone}
* **Best Season:** **{season.upper()}**
* **System Confidence:** {confidence}%

---

### 🚫 Colors to Avoid
{', '.join(palette_data['avoid'])}

---

### 💎 Styling Recommendations
* **Jewelry:** {palette_data['jewelry']}
* **Hair Color:** {palette_data['hair']}
* **Makeup Palette:** {palette_data['makeup']}
"""

    return output_text, palette_img


# ==========================================
# 6. GRADIO INTERFACE SETUP
# ==========================================
interface = gr.Interface(
    fn=analyze_personal_color,
    inputs=gr.Image(type="pil", label="Upload Selfie (Clear, Neutral Lighting)"),
    outputs=[
        gr.Markdown(label="Analysis Results"),
        gr.Image(type="pil", label="Your Recommended Palette Swatches"),
    ],
    title="AI Personal Color Analysis Engine",
    description="Upload a front-facing selfie with natural, balanced lighting. This AI agent extracts dominant skin properties using MediaPipe Face Mesh landmarks and applies a mathematical color classification model to find your seasonal color profile.",
    theme="soft",
)

if __name__ == "__main__":
    interface.launch(inline=True, share=True)


/usr/local/lib/python3.12/dist-packages/gradio/interface.py:171: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  super().__init__(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://1d74f954a0e1465391.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
